In [ ]:
!pip install faker 
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
np.random.seed(42)

In [ ]:
# --- Config ---
NUM_USERS = 500
FEATURES = ["dashboard", "reports", "api_access", "team_collab", "integrations", "export", "notifications"]
PLANS = ["free", "starter", "pro"]

In [ ]:
# --- Generate users ---
users = []
for i in range(NUM_USERS):
    signup = fake.date_between(start_date="-1y", end_date="-30d")
    plan = random.choices(PLANS, weights=[0.5, 0.3, 0.2])[0]
    users.append({
        "user_id": f"U{1000+i}",
        "name": fake.name(),
        "email": fake.email(),
        "plan": plan,
        "signup_date": signup,
        "country": fake.country_code()
    })

users_df = pd.DataFrame(users)

In [ ]:
# --- Generate events ---
events = []
for _, user in users_df.iterrows():
    signup = user["signup_date"]
    # Power users do more, free users do less
    activity_level = {"free": 3, "starter": 10, "pro": 25}[user["plan"]]
    num_events = random.randint(1, activity_level * 10)

    for _ in range(num_events):
        event_date = signup + timedelta(days=random.randint(0, 365))
        if event_date > datetime.today().date():
            continue
        feature = random.choices(
            FEATURES,
            weights=[0.35, 0.25, 0.1, 0.1, 0.08, 0.07, 0.05]
        )[0]
        events.append({
            "user_id": user["user_id"],
            "event": f"used_{feature}",
            "feature": feature,
            "event_date": event_date,
            "plan": user["plan"]
        })

events_df = pd.DataFrame(events)

In [ ]:
# --- Save ---
users_df.to_csv("users.csv", index=False)
events_df.to_csv("events.csv", index=False)
print(f"Generated {len(users_df)} users and {len(events_df)} events")
print(events_df.head())